# Stage 8: Label Shift Experiments

This notebook studies how changing the clean CIFAR-10 evaluation class proportions changes coverage and prediction-set efficiency. The full design enumerates all 45 unordered pairs of emphasized classes at three levels, plus a balanced baseline. Images, model predictions, clean calibration thresholds, and APS/RAPS randomization remain fixed.

The three split seeds describe sensitivity to the calibration/evaluation partition; their mean and sample SD are not independent-replication uncertainty. Cat/dog and airplane/automobile are retained as illustrative contrasts and are checked against the full enumeration.

The first sections reconstruct the class-conditional statistics from cached predictions and verify the saved clean baselines. The full-rotation section can also run independently using the committed class-conditional CSVs, without loading prediction caches. Final figures are in `05_final_analysis.ipynb`.

## Definitions and expected inputs

For method \(m\), model \(f\), split \(s\), and class \(y\), calculate evaluation count, empirical \(P(Y\in C(X)\mid Y=y)\), mean set size, empty-set rate, and top-1 accuracy. These are descriptive class-conditional statistics; they are not conformal class-conditional guarantees.

For target class prior \(\pi\), compute each shifted aggregate as \(\sum_y \pi_y g_y\), where \(g_y\) is that class’s empirical conditional statistic. Prediction sets are calculated once from the existing clean probabilities and thresholds; reweighting changes only their aggregate weights.

Inputs are the two Stage 4 clean prediction caches, three saved split files, the saved LAC/APS/RAPS clean thresholds and clean metrics, and the seed-2026 APS randomization array. Outputs are CSV tables saved under the project’s `results/` directory. The first audit reconstructs each saved clean metric using the **actual** class proportions of that evaluation split; it does not assume every split has exactly 10% per class.

In [1]:
from pathlib import Path
import json
import math
import pickle
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/conformal_shift")
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = current.parent if current.name == "notebooks" else current

CACHE_ROOT = PROJECT_ROOT / "cached_outputs" / "stage4_predictions"
SPLIT_ROOT = PROJECT_ROOT / "data" / "splits"
RESULTS_ROOT = PROJECT_ROOT / "results"
CIFAR_TEST_BATCH = PROJECT_ROOT / "data" / "raw" / "cifar-10-batches-py" / "test_batch"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

ALPHA = 0.10
TARGET_COVERAGE = 1.0 - ALPHA
SPLIT_SEEDS = (42, 43, 44)
MODEL_KEYS = ("standard_resnet18", "augmix_resnet18")
METHODS = ("LAC", "APS", "RAPS")
CLASS_NAMES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
N_TEST = 10_000
N_EVAL = 5_000
APS_UNIFORMS_PATH = RESULTS_ROOT / "aps_randomization_uniforms_seed2026.npy"

print(f"Project root: {PROJECT_ROOT}")
print(f"Clean cache root: {CACHE_ROOT}")
print(f"Saved split root: {SPLIT_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print("No model inference or recalibration will be performed.")

Mounted at /content/drive
Project root: /content/drive/MyDrive/conformal_shift
Clean cache root: /content/drive/MyDrive/conformal_shift/cached_outputs/stage4_predictions
Saved split root: /content/drive/MyDrive/conformal_shift/data/splits
Results root: /content/drive/MyDrive/conformal_shift/results
No model inference or recalibration will be performed.


## Load and validate the frozen inputs

The cache arrays use canonical CIFAR-10 test indices 0–9,999. The checks below confirm cache identity and label alignment, split disjointness, saved randomization availability, and the presence of exactly six existing clean thresholds and clean metric rows per method. Thresholds are loaded from prior results and are never recomputed in this notebook.

In [2]:
def load_clean_cache(model_key):
    path = CACHE_ROOT / model_key / "clean_test.npz"
    if not path.is_file():
        raise FileNotFoundError(f"Missing clean prediction cache: {path}")
    with np.load(path, allow_pickle=False) as stored:
        cache = {name: stored[name] for name in stored.files}
    probabilities = np.asarray(cache["probabilities"])
    labels = np.asarray(cache["labels"], dtype=np.int64)
    indices = np.asarray(cache["original_test_indices"], dtype=np.int64)
    if probabilities.shape != (N_TEST, 10) or labels.shape != (N_TEST,) or indices.shape != (N_TEST,):
        raise ValueError(f"Unexpected clean cache shapes for {model_key}")
    if not np.array_equal(indices, np.arange(N_TEST, dtype=np.int64)):
        raise ValueError(f"Cache indices are not canonical for {model_key}")
    if not np.isfinite(probabilities).all() or np.any(probabilities < 0) or np.any(probabilities > 1):
        raise ValueError(f"Invalid probabilities in {model_key} clean cache")
    if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5, rtol=1e-5):
        raise ValueError(f"Probability rows do not sum to one for {model_key}")
    return {"probabilities": probabilities, "labels": labels, "indices": indices, "path": path}

clean_caches = {model: load_clean_cache(model) for model in MODEL_KEYS}
canonical_labels = clean_caches[MODEL_KEYS[0]]["labels"]
for model in MODEL_KEYS:
    if not np.array_equal(clean_caches[model]["labels"], canonical_labels):
        raise ValueError("Standard and AugMix clean-cache labels do not align")

if CIFAR_TEST_BATCH.is_file():
    with CIFAR_TEST_BATCH.open("rb") as f:
        cifar_test = pickle.load(f, encoding="bytes")
    raw_labels = np.asarray(cifar_test[b"labels"], dtype=np.int64)
    if not np.array_equal(raw_labels, canonical_labels):
        raise ValueError("Clean cache labels do not match the canonical CIFAR-10 test labels")
    print("Clean-cache labels match the canonical CIFAR-10 test batch: PASS")
else:
    print("Canonical test batch is not present at this project path; cross-model cached-label agreement will be used.")

splits = {}
all_indices = np.arange(N_TEST, dtype=np.int64)
for seed in SPLIT_SEEDS:
    path = SPLIT_ROOT / f"clean_test_split_seed{seed}.npz"
    with np.load(path, allow_pickle=False) as stored:
        cal = np.asarray(stored["calibration_indices"], dtype=np.int64)
        ev = np.asarray(stored["evaluation_indices"], dtype=np.int64)
    if cal.shape != (N_EVAL,) or ev.shape != (N_EVAL,) or np.intersect1d(cal, ev).size:
        raise ValueError(f"Split {seed} is not a disjoint 5,000/5,000 split")
    if not np.array_equal(np.sort(np.concatenate([cal, ev])), all_indices):
        raise ValueError(f"Split {seed} does not cover canonical indices 0..9999")
    splits[seed] = {"calibration_indices": cal, "evaluation_indices": ev}
    print(f"Split {seed}: disjoint, 5,000 evaluation examples: PASS")

if not APS_UNIFORMS_PATH.is_file():
    raise FileNotFoundError(f"Missing saved seed-2026 APS/RAPS randomization values: {APS_UNIFORMS_PATH}")
aps_uniform_by_index = np.load(APS_UNIFORMS_PATH, allow_pickle=False)
if aps_uniform_by_index.shape != (N_TEST,) or not np.isfinite(aps_uniform_by_index).all() or np.any((aps_uniform_by_index < 0) | (aps_uniform_by_index >= 1)):
    raise ValueError("Saved APS/RAPS randomization array failed its shape/range check")

thresholds = {}
saved_clean_metrics = {}
for method, prefix in (("LAC", "lac"), ("APS", "aps"), ("RAPS", "raps")):
    threshold_path = RESULTS_ROOT / f"{prefix}_calibration_thresholds.csv"
    clean_path = RESULTS_ROOT / f"{prefix}_clean_metrics.csv"
    threshold_table = pd.read_csv(threshold_path)
    clean_table = pd.read_csv(clean_path)
    if len(threshold_table) != 6 or len(clean_table) != 6:
        raise ValueError(f"Expected six existing clean thresholds and clean metric rows for {method}")
    if not {"model_key", "split_seed", "q_hat"}.issubset(threshold_table.columns):
        raise ValueError(f"Unexpected threshold table columns for {method}: {threshold_table.columns.tolist()}")
    if not {"model_key", "split_seed", "empirical_coverage", "average_set_size", "empty_set_proportion", "top1_accuracy"}.issubset(clean_table.columns):
        raise ValueError(f"Unexpected clean metric table columns for {method}: {clean_table.columns.tolist()}")
    for row in threshold_table.itertuples():
        key = (method, str(row.model_key), int(row.split_seed))
        thresholds[key] = float(row.q_hat)
    saved_clean_metrics[method] = clean_table
    if len(thresholds) < 6 * (METHODS.index(method) + 1):
        raise ValueError(f"Threshold loading incomplete for {method}")

expected_keys = {(method, model, seed) for method in METHODS for model in MODEL_KEYS for seed in SPLIT_SEEDS}
if set(thresholds) != expected_keys:
    raise ValueError("Saved clean threshold keys do not match the expected methods/models/splits")
print("Loaded 18 existing clean thresholds and seed-2026 randomization values; no threshold was recalibrated.")

Clean-cache labels match the canonical CIFAR-10 test batch: PASS
Split 42: disjoint, 5,000 evaluation examples: PASS
Split 43: disjoint, 5,000 evaluation examples: PASS
Split 44: disjoint, 5,000 evaluation examples: PASS
Loaded 18 existing clean thresholds and seed-2026 randomization values; no threshold was recalibrated.


## Recreate the validated prediction-set rules

These small functions apply the already validated LAC, randomized APS, and randomized RAPS set definitions to cached clean probabilities. They do not estimate thresholds. APS and RAPS use the saved uniform value associated with each canonical image index, shared across candidate labels.

In [3]:
RAPS_LAMBDA = 0.02
RAPS_K_REG = 1

def lac_prediction_sets(probabilities, q_hat):
    p = np.asarray(probabilities, dtype=np.float64)
    return (1.0 - p) <= q_hat

def aps_prediction_sets(probabilities, q_hat, uniforms):
    p = np.asarray(probabilities, dtype=np.float64)
    u = np.asarray(uniforms, dtype=np.float64)
    if p.ndim != 2 or u.shape != (len(p),):
        raise ValueError("Expected probabilities (N,C) and one uniform per image (N,)")
    order = np.argsort(-p, axis=1, kind="stable")
    sorted_p = np.take_along_axis(p, order, axis=1)
    before = np.cumsum(sorted_p, axis=1) - sorted_p
    in_order = before + u[:, None] * sorted_p <= q_hat
    membership = np.zeros_like(in_order, dtype=bool)
    np.put_along_axis(membership, order, in_order, axis=1)
    return membership

def raps_prediction_sets(probabilities, q_hat, uniforms, lambda_reg=RAPS_LAMBDA, k_reg=RAPS_K_REG):
    p = np.asarray(probabilities, dtype=np.float64)
    u = np.asarray(uniforms, dtype=np.float64)
    if p.ndim != 2 or u.shape != (len(p),):
        raise ValueError("Expected probabilities (N,C) and one uniform per image (N,)")
    order = np.argsort(-p, axis=1, kind="stable")
    sorted_p = np.take_along_axis(p, order, axis=1)
    before = np.cumsum(sorted_p, axis=1) - sorted_p
    ranks_1_based = np.broadcast_to(np.arange(1, p.shape[1] + 1), p.shape)
    sorted_scores = before + u[:, None] * sorted_p + lambda_reg * np.maximum(ranks_1_based - k_reg, 0)
    in_order = sorted_scores <= q_hat
    membership = np.zeros_like(in_order, dtype=bool)
    np.put_along_axis(membership, order, in_order, axis=1)
    return membership

# Quick definition checks: shared U, inclusive threshold boundary, and 1-based rank penalty.
toy_p = np.asarray([[0.50, 0.30, 0.15, 0.05]])
toy_u = np.asarray([0.50])
assert np.flatnonzero(aps_prediction_sets(toy_p, 0.65, toy_u)[0]).tolist() == [0, 1]
assert np.flatnonzero(raps_prediction_sets(toy_p, 0.67, toy_u)[0]).tolist() == [0, 1]
assert np.array_equal(lac_prediction_sets(toy_p, 0.70), np.asarray([[True, True, False, False]]))
tie_p = np.asarray([[0.40, 0.40, 0.20]])
assert np.argsort(-tie_p, axis=1, kind="stable")[0].tolist() == [0, 1, 2]
print("Cached-probability set rules and hand-checkable boundaries: PASS")

Cached-probability set rules and hand-checkable boundaries: PASS


## Class-conditional statistics and baseline reconstruction gate

For each clean evaluation split, calculate per-image coverage, set size, empty-set indicator, and top-1 correctness once. Aggregate these within each true class. Then reconstruct the saved aggregate clean metrics with the split’s actual empirical class proportions.

Each split is drawn from the balanced CIFAR-10 test set but its finite evaluation subset need not contain exactly 500 images per class. Therefore the reconstruction uses \(n_y/5000\) for each class. The later “Balanced” scenario deliberately uses the target prior of exactly 0.10 per class and can differ slightly from the saved aggregate clean score.

**Gate:** every method/model/split/metric reconstruction must match the saved clean metric within \(10^{-10}\). If any comparison fails, inspect indices, thresholds, saved randomization, and metric definitions; do not run the scenario cell.

In [4]:
metric_names = ("coverage", "average_set_size", "empty_set_rate", "top1_accuracy")
class_records = []
per_image = {}

for method in METHODS:
    for model in MODEL_KEYS:
        cache = clean_caches[model]
        p_all = cache["probabilities"]
        y_all = cache["labels"]
        for seed in SPLIT_SEEDS:
            eval_idx = splits[seed]["evaluation_indices"]
            p = p_all[eval_idx]
            y = y_all[eval_idx]
            u = aps_uniform_by_index[eval_idx]
            q = thresholds[(method, model, seed)]
            if method == "LAC":
                sets = lac_prediction_sets(p, q)
            elif method == "APS":
                sets = aps_prediction_sets(p, q, u)
            else:
                sets = raps_prediction_sets(p, q, u)
            sizes = sets.sum(axis=1).astype(np.int64)
            covered = sets[np.arange(len(y)), y]
            correct = (p.argmax(axis=1) == y)
            values = {
                "coverage": covered.astype(np.float64),
                "average_set_size": sizes.astype(np.float64),
                "empty_set_rate": (sizes == 0).astype(np.float64),
                "top1_accuracy": correct.astype(np.float64),
            }
            per_image[(method, model, seed)] = {"labels": y, **values, "set_sizes": sizes}
            for cls, class_name in enumerate(CLASS_NAMES):
                mask = y == cls
                count = int(mask.sum())
                if count == 0:
                    raise ValueError(f"Evaluation split {seed} contains no examples of class {class_name}")
                class_records.append({
                    "method": method, "model_key": model, "split_seed": seed,
                    "class_id": cls, "class_name": class_name, "evaluation_count": count,
                    "class_coverage": float(values["coverage"][mask].mean()),
                    "class_average_set_size": float(values["average_set_size"][mask].mean()),
                    "class_empty_set_rate": float(values["empty_set_rate"][mask].mean()),
                    "class_top1_accuracy": float(values["top1_accuracy"][mask].mean()),
                })

class_conditional = pd.DataFrame(class_records)
assert len(class_conditional) == 2 * 3 * 3 * 10
print("Class-conditional rows (180 expected):", len(class_conditional))
display(class_conditional.head(20))

baseline_records = []
for method in METHODS:
    saved = saved_clean_metrics[method]
    for model in MODEL_KEYS:
        for seed in SPLIT_SEEDS:
            rows = class_conditional[(class_conditional.method == method) &
                                     (class_conditional.model_key == model) &
                                     (class_conditional.split_seed == seed)].sort_values("class_id")
            actual_prior = rows["evaluation_count"].to_numpy(dtype=np.float64) / N_EVAL
            reconstructed = {
                "empirical_coverage": float(np.dot(actual_prior, rows["class_coverage"])),
                "average_set_size": float(np.dot(actual_prior, rows["class_average_set_size"])),
                "empty_set_proportion": float(np.dot(actual_prior, rows["class_empty_set_rate"])),
                "top1_accuracy": float(np.dot(actual_prior, rows["class_top1_accuracy"])),
            }
            saved_row = saved[(saved.model_key == model) & (saved.split_seed.astype(int) == seed)]
            if len(saved_row) != 1:
                raise ValueError(f"Expected one saved clean metric row for {method}/{model}/{seed}")
            saved_row = saved_row.iloc[0]
            comparisons = {}
            for metric, rebuilt in reconstructed.items():
                error = abs(rebuilt - float(saved_row[metric]))
                comparisons[metric + "_abs_error"] = error
                if error > 1e-10:
                    raise RuntimeError(f"Baseline reconstruction failed for {method}/{model}/{seed}/{metric}: {error}")
            baseline_records.append({
                "method": method, "model_key": model, "split_seed": seed,
                "actual_class_proportions": json.dumps(actual_prior.tolist()),
                **{"reconstructed_" + k: v for k, v in reconstructed.items()},
                **comparisons, "matches_saved_metrics": True,
            })

baseline_audit = pd.DataFrame(baseline_records)
assert len(baseline_audit) == 18 and baseline_audit["matches_saved_metrics"].all()
print("Baseline reconstruction: 18/18 method/model/split rows; four metrics agree within 1e-10: PASS")
display(baseline_audit[["method", "model_key", "split_seed", "matches_saved_metrics"]])
class_conditional.to_csv(RESULTS_ROOT / "label_shift_class_conditional_metrics.csv", index=False)
baseline_audit.to_csv(RESULTS_ROOT / "label_shift_baseline_reconstruction_audit.csv", index=False)
print("Saved class-conditional metrics and baseline audit. Scenario evaluation is gated on the passed audit.")

Class-conditional rows (180 expected): 180


,method,model_key,split_seed,class_id,class_name,evaluation_count,class_coverage,class_average_set_size,class_empty_set_rate,class_top1_accuracy
0,LAC,standard_resnet18,42,0,airplane,473,0.892178,0.900634,0.099366,0.955603
1,LAC,standard_resnet18,42,1,automobile,490,0.928571,0.942857,0.057143,0.967347
2,LAC,standard_resnet18,42,2,bird,498,0.835341,0.855422,0.144578,0.931727
3,LAC,standard_resnet18,42,3,cat,514,0.795720,0.848249,0.151751,0.883268
4,LAC,standard_resnet18,42,4,deer,505,0.914851,0.922772,0.077228,0.968317
5,LAC,standard_resnet18,42,5,dog,501,0.854291,0.872255,0.127745,0.922156
6,LAC,standard_resnet18,42,6,frog,512,0.917969,0.921875,0.078125,0.968750
7,LAC,standard_resnet18,42,7,horse,510,0.931373,0.941176,0.058824,0.966667
8,LAC,standard_resnet18,42,8,ship,500,0.934000,0.944000,0.056000,0.970000
9,LAC,standard_resnet18,42,9,truck,497,0.935614,0.943662,0.056338,0.973843


Baseline reconstruction: 18/18 method/model/split rows; four metrics agree within 1e-10: PASS


,method,model_key,split_seed,matches_saved_metrics
0,LAC,standard_resnet18,42,True
1,LAC,standard_resnet18,43,True
2,LAC,standard_resnet18,44,True
3,LAC,augmix_resnet18,42,True
4,LAC,augmix_resnet18,43,True
5,LAC,augmix_resnet18,44,True
6,APS,standard_resnet18,42,True
7,APS,standard_resnet18,43,True
8,APS,standard_resnet18,44,True
9,APS,augmix_resnet18,42,True


Saved class-conditional metrics and baseline audit. Scenario evaluation is gated on the passed audit.


## Two illustrative class-pair scenarios

These seven priors retain balanced, cat/dog-heavy, and airplane/automobile-heavy contrasts for the detailed example plots. At mild, moderate, and severe levels, the emphasized pair receives total mass 0.30, 0.40, and 0.60; the remaining eight classes share the rest equally. The full-rotation section below includes these cases and every other pair under the same definitions.

In [5]:
if len(baseline_audit) != 18 or not baseline_audit["matches_saved_metrics"].all():
    raise RuntimeError("Baseline reconstruction gate has not passed; do not evaluate shifted scenarios")

scenario_priors = {"balanced": np.full(10, 0.10, dtype=np.float64)}
for severity, focus_mass in (("mild", 0.30), ("moderate", 0.40), ("severe", 0.60)):
    other_mass = (1.0 - focus_mass) / 8.0
    cat_dog = np.full(10, other_mass, dtype=np.float64)
    cat_dog[[3, 5]] = focus_mass / 2.0
    air_auto = np.full(10, other_mass, dtype=np.float64)
    air_auto[[0, 1]] = focus_mass / 2.0
    scenario_priors[f"cat_dog_{severity}"] = cat_dog
    scenario_priors[f"airplane_automobile_{severity}"] = air_auto

prior_records = []
for scenario, prior in scenario_priors.items():
    if prior.shape != (10,) or not np.isfinite(prior).all() or np.any(prior < 0) or not np.isclose(prior.sum(), 1.0, atol=1e-12):
        raise ValueError(f"Invalid class prior for scenario {scenario}")
    for cls, probability in enumerate(prior):
        prior_records.append({"scenario": scenario, "class_id": cls, "class_name": CLASS_NAMES[cls], "target_probability": float(probability)})
prior_table = pd.DataFrame(prior_records)
print("Fixed target priors (all sum to one):")
display(prior_table.pivot(index="scenario", columns="class_name", values="target_probability").reindex(columns=CLASS_NAMES))

Fixed target priors (all sum to one):


class_name,airplane,automobile,bird,cat,deer,dog,frog,horse,ship,truck
scenario,,,,,,,,,,
airplane_automobile_mild,0.1500,0.1500,0.0875,0.0875,0.0875,0.0875,0.0875,0.0875,0.0875,0.0875
airplane_automobile_moderate,0.2000,0.2000,0.0750,0.0750,0.0750,0.0750,0.0750,0.0750,0.0750,0.0750
airplane_automobile_severe,0.3000,0.3000,0.0500,0.0500,0.0500,0.0500,0.0500,0.0500,0.0500,0.0500
balanced,0.1000,0.1000,0.1000,0.1000,0.1000,0.1000,0.1000,0.1000,0.1000,0.1000
cat_dog_mild,0.0875,0.0875,0.0875,0.1500,0.0875,0.1500,0.0875,0.0875,0.0875,0.0875
cat_dog_moderate,0.0750,0.0750,0.0750,0.2000,0.0750,0.2000,0.0750,0.0750,0.0750,0.0750
cat_dog_severe,0.0500,0.0500,0.0500,0.3000,0.0500,0.3000,0.0500,0.0500,0.0500,0.0500


## Analytical shifted metrics, Monte Carlo check, and paired summaries

For each saved split, method, model, and scenario, combine the class-conditional empirical statistics using the fixed target prior. No image, prediction set, threshold, or randomization value is changed.

The Monte Carlo check is prespecified as Standard ResNet-18 + APS + seed 42 under the cat/dog moderate prior. It samples 5,000 classes from that prior and then samples evaluation images with replacement within each selected class. The analytical estimate and standard error come from the same empirical class-conditional population. A fixed RNG seed makes the check reproducible; the Monte Carlo result is not expected to equal the analytical estimate exactly.

In [6]:
shift_records = []
for method in METHODS:
    for model in MODEL_KEYS:
        for seed in SPLIT_SEEDS:
            class_rows = class_conditional[(class_conditional.method == method) &
                                           (class_conditional.model_key == model) &
                                           (class_conditional.split_seed == seed)].sort_values("class_id")
            for scenario, prior in scenario_priors.items():
                metric_sources = {
                    "empirical_coverage": "class_coverage",
                    "average_set_size": "class_average_set_size",
                    "empty_set_rate": "class_empty_set_rate",
                    "top1_accuracy": "class_top1_accuracy",
                }
                row = {"method": method, "model_key": model, "split_seed": seed, "scenario": scenario}
                for output_name, source in metric_sources.items():
                    row[output_name] = float(np.dot(prior, class_rows[source].to_numpy(dtype=np.float64)))
                row["coverage_gap"] = TARGET_COVERAGE - row["empirical_coverage"]
                row["set_size_meaning_note"] = "Aggregate set size; individual image prediction sets are unchanged."
                shift_records.append(row)

label_shift_results = pd.DataFrame(shift_records)
assert len(label_shift_results) == 2 * 3 * 3 * len(scenario_priors)
assert np.isfinite(label_shift_results[["empirical_coverage", "coverage_gap", "average_set_size", "empty_set_rate", "top1_accuracy"]].to_numpy()).all()

summary_metrics = ["empirical_coverage", "coverage_gap", "average_set_size", "empty_set_rate", "top1_accuracy"]
label_shift_summary = label_shift_results.groupby(["method", "model_key", "scenario"])[summary_metrics].agg(["mean", "std"]).reset_index()
label_shift_summary.columns = ["_".join(str(part) for part in col if part != "") if isinstance(col, tuple) else col for col in label_shift_summary.columns]
label_shift_summary = label_shift_summary.rename(columns={"method_":"method", "model_key_":"model_key", "scenario_":"scenario"})

# Paired method comparison on identical model/split/scenario keys.
paired_methods = label_shift_results.pivot(index=["model_key", "split_seed", "scenario"], columns="method", values=summary_metrics).reset_index()
paired_methods.columns = ["_".join(str(part) for part in col if part != "") if isinstance(col, tuple) else col for col in paired_methods.columns]
# The pivot produces one row per exact comparison key; validate the full key count.
assert len(paired_methods) == 2 * 3 * len(scenario_priors)

# Paired standard-vs-AugMix comparison on identical method/split/scenario keys.
paired_models = label_shift_results.pivot(index=["method", "split_seed", "scenario"], columns="model_key", values=summary_metrics).reset_index()
paired_models.columns = ["_".join(str(part) for part in col if part != "") if isinstance(col, tuple) else col for col in paired_models.columns]
assert len(paired_models) == 3 * 3 * len(scenario_priors)
for metric in summary_metrics:
    standard_col = metric + "_standard_resnet18"
    augmix_col = metric + "_augmix_resnet18"
    paired_models[metric + "_augmix_minus_standard"] = paired_models[augmix_col] - paired_models[standard_col]

# Monte Carlo check: fixed scenario/method/model/split chosen before looking at results.
mc_method, mc_model, mc_seed, mc_scenario = "APS", "standard_resnet18", 42, "cat_dog_moderate"
mc_prior = scenario_priors[mc_scenario]
mc_image = per_image[(mc_method, mc_model, mc_seed)]
mc_labels = mc_image["labels"]
mc_rng = np.random.default_rng(20261010)
sampled_classes = mc_rng.choice(np.arange(10), size=N_EVAL, replace=True, p=mc_prior)
mc_estimates = {name: np.empty(N_EVAL, dtype=np.float64) for name in metric_sources}
for cls in range(10):
    out_pos = np.flatnonzero(sampled_classes == cls)
    if len(out_pos) == 0:
        continue
    class_pool = np.flatnonzero(mc_labels == cls)
    chosen = mc_rng.choice(class_pool, size=len(out_pos), replace=True)
    mc_estimates["empirical_coverage"][out_pos] = mc_image["coverage"][chosen]
    mc_estimates["average_set_size"][out_pos] = mc_image["average_set_size"][chosen]
    mc_estimates["empty_set_rate"][out_pos] = mc_image["empty_set_rate"][chosen]
    mc_estimates["top1_accuracy"][out_pos] = mc_image["top1_accuracy"][chosen]

analytical_row = label_shift_results[(label_shift_results.method == mc_method) &
                                     (label_shift_results.model_key == mc_model) &
                                     (label_shift_results.split_seed == mc_seed) &
                                     (label_shift_results.scenario == mc_scenario)].iloc[0]
mc_rows = []
for metric in ("empirical_coverage", "average_set_size", "empty_set_rate", "top1_accuracy"):
    vals = mc_image["coverage" if metric == "empirical_coverage" else metric]
    class_means = np.asarray([vals[mc_labels == cls].mean() for cls in range(10)])
    class_second_moments = np.asarray([(vals[mc_labels == cls] ** 2).mean() for cls in range(10)])
    analytical = float(np.dot(mc_prior, class_means))
    mixture_variance = max(0.0, float(np.dot(mc_prior, class_second_moments)) - analytical ** 2)
    expected_se = math.sqrt(mixture_variance / N_EVAL)
    monte_carlo = float(mc_estimates[metric].mean())
    difference = monte_carlo - analytical
    mc_rows.append({"metric": metric, "analytical_weighted_estimate": analytical,
                    "monte_carlo_estimate": monte_carlo, "difference": difference,
                    "expected_sampling_se": expected_se,
                    "absolute_difference_in_se": abs(difference) / expected_se if expected_se > 0 else 0.0})
monte_carlo_audit = pd.DataFrame(mc_rows)
if (monte_carlo_audit["absolute_difference_in_se"] > 5.0).any():
    raise RuntimeError("Monte Carlo validation exceeded five expected standard errors; inspect resampling and weighting before interpreting shifts")

# Save all machine-readable results only after baseline and Monte Carlo checks pass.
prior_table.to_csv(RESULTS_ROOT / "label_shift_target_priors.csv", index=False)
label_shift_results.to_csv(RESULTS_ROOT / "label_shift_split_results.csv", index=False)
label_shift_summary.to_csv(RESULTS_ROOT / "label_shift_summary_mean_std.csv", index=False)
paired_methods.to_csv(RESULTS_ROOT / "label_shift_method_comparison_paired.csv", index=False)
paired_models.to_csv(RESULTS_ROOT / "label_shift_model_comparison_paired.csv", index=False)
monte_carlo_audit.to_csv(RESULTS_ROOT / "label_shift_monte_carlo_validation.csv", index=False)

print("Analytical label-shift results (split-specific rows):", len(label_shift_results))
display(label_shift_results.head(15))
print("Mean and sample SD across the three overlapping split sensitivity checks:")
display(label_shift_summary)
print("Paired LAC / APS / RAPS comparisons: exact same model, split, and scenario keys.")
display(paired_methods.head(12))
print("Paired Standard / AugMix comparisons: exact same method, split, and scenario keys.")
display(paired_models.head(12))
print("Fixed-seed class-conditional resampling check; expected SE shown for sampling variability:")
display(monte_carlo_audit)
print("Saved Stage 8 CSV tables under:", RESULTS_ROOT)

Analytical label-shift results (split-specific rows): 126


,method,model_key,split_seed,scenario,empirical_coverage,average_set_size,empty_set_rate,top1_accuracy,coverage_gap,set_size_meaning_note
0,LAC,standard_resnet18,42,balanced,0.893991,0.909290,0.090710,0.950768,0.006009,Aggregate set size; individual image predictio...
1,LAC,standard_resnet18,42,cat_dog_mild,0.885368,0.903161,0.096839,0.944761,0.014632,Aggregate set size; individual image predictio...
2,LAC,standard_resnet18,42,airplane_automobile_mild,0.896039,0.910847,0.089153,0.952106,0.003961,Aggregate set size; individual image predictio...
3,LAC,standard_resnet18,42,cat_dog_moderate,0.876745,0.897031,0.102969,0.938754,0.023255,Aggregate set size; individual image predictio...
4,LAC,standard_resnet18,42,airplane_automobile_moderate,0.898087,0.912404,0.087596,0.953444,0.001913,Aggregate set size; individual image predictio...
5,LAC,standard_resnet18,42,cat_dog_severe,0.859498,0.884771,0.115229,0.926740,0.040502,Aggregate set size; individual image predictio...
6,LAC,standard_resnet18,42,airplane_automobile_severe,0.902183,0.915518,0.084482,0.956121,-0.002183,Aggregate set size; individual image predictio...
7,LAC,standard_resnet18,43,balanced,0.904004,0.920562,0.079438,0.951395,-0.004004,Aggregate set size; individual image predictio...
8,LAC,standard_resnet18,43,cat_dog_mild,0.894272,0.913978,0.086022,0.945522,0.005728,Aggregate set size; individual image predictio...
9,LAC,standard_resnet18,43,airplane_automobile_mild,0.907609,0.923317,0.076683,0.953641,-0.007609,Aggregate set size; individual image predictio...


Mean and sample SD across the three overlapping split sensitivity checks:


,method,model_key,scenario,empirical_coverage_mean,empirical_coverage_std,coverage_gap_mean,coverage_gap_std,average_set_size_mean,average_set_size_std,empty_set_rate_mean,empty_set_rate_std,top1_accuracy_mean,top1_accuracy_std
0,APS,augmix_resnet18,airplane_automobile_mild,0.901918,0.010843,-0.001918,0.010843,1.022892,0.014693,0.076814,0.009831,0.955364,0.000479
1,APS,augmix_resnet18,airplane_automobile_moderate,0.901471,0.011521,-0.001471,0.011521,1.016307,0.015730,0.078323,0.010699,0.957941,0.000594
2,APS,augmix_resnet18,airplane_automobile_severe,0.900578,0.013096,-0.000578,0.013096,1.003137,0.017884,0.081341,0.012497,0.963096,0.000872
3,APS,augmix_resnet18,balanced,0.902364,0.010258,-0.002364,0.010258,1.029477,0.013689,0.075305,0.008993,0.952787,0.000399
4,APS,augmix_resnet18,cat_dog_mild,0.902096,0.009635,-0.002096,0.009635,1.045819,0.014341,0.073036,0.008390,0.946550,0.001293
5,APS,augmix_resnet18,cat_dog_moderate,0.901827,0.009012,-0.001827,0.009012,1.062160,0.015112,0.070767,0.007787,0.940312,0.002188
6,APS,augmix_resnet18,cat_dog_severe,0.901289,0.007770,-0.001289,0.007770,1.094843,0.016946,0.066228,0.006584,0.927838,0.003978
7,APS,standard_resnet18,airplane_automobile_mild,0.901976,0.005908,-0.001976,0.005908,0.992020,0.004852,0.067799,0.006819,0.952332,0.001212
8,APS,standard_resnet18,airplane_automobile_moderate,0.902296,0.006792,-0.002296,0.006792,0.989543,0.005492,0.068666,0.007391,0.953970,0.001716
9,APS,standard_resnet18,airplane_automobile_severe,0.902937,0.008572,-0.002937,0.008572,0.984590,0.006856,0.070398,0.008537,0.957247,0.002748


Paired LAC / APS / RAPS comparisons: exact same model, split, and scenario keys.


...

Paired Standard / AugMix comparisons: exact same method, split, and scenario keys.


,method,split_seed,scenario,empirical_coverage_augmix_resnet18,empirical_coverage_standard_resnet18,coverage_gap_augmix_resnet18,coverage_gap_standard_resnet18,average_set_size_augmix_resnet18,average_set_size_standard_resnet18,empty_set_rate_augmix_resnet18,empty_set_rate_standard_resnet18,top1_accuracy_augmix_resnet18,top1_accuracy_standard_resnet18,empirical_coverage_augmix_minus_standard,coverage_gap_augmix_minus_standard,average_set_size_augmix_minus_standard,empty_set_rate_augmix_minus_standard,top1_accuracy_augmix_minus_standard
0,APS,42,airplane_automobile_mild,0.892633,0.896353,0.007367,0.003647,1.010350,0.987715,0.085533,0.074174,0.955833,0.952106,-0.003720,0.003720,0.022635,0.011360,0.003727
1,APS,42,airplane_automobile_moderate,0.890867,0.895706,0.009133,0.004294,1.002342,0.985034,0.088196,0.075621,0.958430,0.953444,-0.004839,0.004839,0.017308,0.012575,0.004986
2,APS,42,airplane_automobile_severe,0.887334,0.894412,0.012666,0.005588,0.986326,0.979672,0.093520,0.078515,0.963625,0.956121,-0.007078,0.007078,0.006655,0.015005,0.007503
3,APS,42,balanced,0.894399,0.897000,0.005601,0.003000,1.018358,0.990396,0.082871,0.072726,0.953236,0.950768,-0.002601,0.002601,0.027962,0.010145,0.002468
4,APS,42,cat_dog_mild,0.894677,0.894402,0.005323,0.005598,1.033097,0.995036,0.080143,0.071503,0.948024,0.944761,0.000275,-0.000275,0.038061,0.008640,0.003264
5,APS,42,cat_dog_moderate,0.894955,0.891805,0.005045,0.008195,1.047837,0.999676,0.077414,0.070279,0.942813,0.938754,0.003150,-0.003150,0.048161,0.007135,0.004059
6,APS,42,cat_dog_severe,0.895511,0.886610,0.004489,0.013390,1.077316,1.008956,0.071957,0.067832,0.932389,0.926740,0.008901,-0.008901,0.068360,0.004125,0.005649
7,APS,43,airplane_automobile_mild,0.899286,0.901441,0.000714,-0.001441,1.019270,0.991067,0.078751,0.068616,0.955384,0.953641,-0.002155,0.002155,0.028203,0.010135,0.001743
8,APS,43,airplane_automobile_moderate,0.899817,0.901909,0.000183,-0.001909,1.013232,0.987937,0.079819,0.069472,0.958114,0.955887,-0.002092,0.002092,0.025295,0.010348,0.002227
9,APS,43,airplane_automobile_severe,0.900879,0.902844,-0.000879,-0.002844,1.001156,0.981677,0.081956,0.071184,0.963574,0.960379,-0.001965,0.001965,0.019479,0.010773,0.003195


Fixed-seed class-conditional resampling check; expected SE shown for sampling variability:


,metric,analytical_weighted_estimate,monte_carlo_estimate,difference,expected_sampling_se,absolute_difference_in_se
0,empirical_coverage,0.891805,0.8880,-0.003805,0.004393,0.866167
1,average_set_size,0.999676,0.9966,-0.003076,0.005833,0.527316
2,empty_set_rate,0.070279,0.0714,0.001121,0.003615,0.309989
3,top1_accuracy,0.938754,0.9392,0.000446,0.003391,0.131581


Saved Stage 8 CSV tables under: /content/drive/MyDrive/conformal_shift/results


## Full rotation over all 45 class pairs

Enumerate every unordered pair of CIFAR-10 classes. A balanced baseline assigns 0.10 to each class. For each of the 45 pairs, mild, moderate, and severe shifts give the pair total probability 0.30, 0.40, and 0.60, divided equally; the other eight classes share the remaining mass equally. This gives 136 target priors including balanced, evaluated for both models, all three methods, and all three split seeds.

This section can run independently from the repository root or `notebooks/`. It reads the saved class-conditional statistics, baseline audit, and illustrative two-pair results under `results/`. It performs CPU reweighting and checks that the balanced and two displayed contrasts reproduce all 126 existing rows. Prediction sets, thresholds, split indices, and model outputs stay fixed.

For each scenario, summarize the three overlapping splits with mean and sample SD. Then describe the distribution of the 45 pair means by level. Each class appears in nine pairs, so the equally weighted average prior is exactly uniform: all-pair mean metrics equal balanced metrics at every level. Pair ranges, quantiles, and counts below 0.90 show sensitivity that the mean cancels. They are descriptive summaries, not confidence intervals or tests of conformal validity.

In [ ]:
from itertools import combinations
from pathlib import Path
import hashlib
import json
import time

import numpy as np
import pandas as pd
from IPython.display import display

rotation_started = time.perf_counter()
rotation_root = next(
    (p.resolve() for p in (Path.cwd(), Path.cwd().parent)
     if (p / "results" / "label_shift_class_conditional_metrics.csv").is_file()),
    None,
)
if rotation_root is None:
    raise FileNotFoundError("Run from the repository root or its notebooks directory.")
rotation_dir = rotation_root / "results"
rotation_inputs = [
    "label_shift_class_conditional_metrics.csv",
    "label_shift_baseline_reconstruction_audit.csv",
    "label_shift_split_results.csv",
]
rotation_classes = pd.read_csv(rotation_dir / rotation_inputs[0])
rotation_baseline_audit = pd.read_csv(rotation_dir / rotation_inputs[1])
rotation_examples = pd.read_csv(rotation_dir / rotation_inputs[2])
rotation_keys = ["method", "model_key", "split_seed"]
rotation_metrics = {
    "empirical_coverage": "class_coverage",
    "average_set_size": "class_average_set_size",
    "empty_set_rate": "class_empty_set_rate",
    "top1_accuracy": "class_top1_accuracy",
}
assert len(rotation_classes) == 180 and len(rotation_baseline_audit) == 18
assert rotation_baseline_audit["matches_saved_metrics"].all()
assert not rotation_classes.duplicated(rotation_keys + ["class_id"]).any()
assert set(rotation_classes.method) == {"LAC", "APS", "RAPS"}
assert set(rotation_classes.model_key) == {"standard_resnet18", "augmix_resnet18"}
assert set(rotation_classes.split_seed) == {42, 43, 44}
rotation_names = (
    rotation_classes[["class_id", "class_name"]].drop_duplicates()
    .sort_values("class_id").class_name.tolist()
)
assert len(rotation_names) == 10
rotation_pairs = list(combinations(range(10), 2))
rotation_levels = {"mild": 0.30, "moderate": 0.40, "severe": 0.60}
rotation_scenarios = [{
    "scenario": "balanced", "pair_class_1": -1, "pair_class_2": -1,
    "pair_name": "balanced", "severity": "balanced", "focus_mass": 0.20,
    "prior": np.full(10, 0.10),
}]
for first, second in rotation_pairs:
    for severity, mass in rotation_levels.items():
        prior = np.full(10, (1.0 - mass) / 8.0)
        prior[[first, second]] = mass / 2.0
        assert np.isclose(prior.sum(), 1.0) and (prior >= 0).all()
        rotation_scenarios.append({
            "scenario": f"pair_{first}_{second}_{severity}",
            "pair_class_1": first, "pair_class_2": second,
            "pair_name": f"{rotation_names[first]} + {rotation_names[second]}",
            "severity": severity, "focus_mass": mass, "prior": prior,
        })
assert len(rotation_pairs) == 45 and len(rotation_scenarios) == 136
for severity in rotation_levels:
    priors = np.stack([s["prior"] for s in rotation_scenarios if s["severity"] == severity])
    assert priors.shape == (45, 10)
    assert np.allclose(priors.mean(axis=0), 0.10, rtol=0, atol=1e-14)

rotation_prior_rows, rotation_rows = [], []
for scenario in rotation_scenarios:
    identity = {key: value for key, value in scenario.items() if key != "prior"}
    for class_id, probability in enumerate(scenario["prior"]):
        rotation_prior_rows.append({
            **identity, "class_id": class_id, "class_name": rotation_names[class_id],
            "target_probability": float(probability),
        })
for group, frame in rotation_classes.groupby(rotation_keys, sort=True):
    frame = frame.sort_values("class_id")
    assert frame.class_id.tolist() == list(range(10))
    assert (frame.evaluation_count > 0).all() and frame.evaluation_count.sum() == 5000
    class_values = frame[list(rotation_metrics.values())].to_numpy(dtype=float)
    assert np.isfinite(class_values).all()
    balanced_values = np.full(10, 0.10) @ class_values
    for scenario in rotation_scenarios:
        values = scenario["prior"] @ class_values
        identity = {key: value for key, value in scenario.items() if key != "prior"}
        row = dict(zip(rotation_keys, group))
        row.update(identity)
        row.update(zip(rotation_metrics, map(float, values)))
        row["coverage_gap"] = 0.90 - row["empirical_coverage"]
        row["coverage_change_from_balanced"] = float(values[0] - balanced_values[0])
        rotation_rows.append(row)
rotation_priors = pd.DataFrame(rotation_prior_rows)
rotation_split = pd.DataFrame(rotation_rows)
assert len(rotation_priors) == 1360 and len(rotation_split) == 2448
assert not rotation_split.duplicated(rotation_keys + ["scenario"]).any()

# The two displayed examples and balanced baseline must reproduce the saved experiment.
example_to_rotation = {"balanced": "balanced"}
for family, pair in {"cat_dog": (3, 5), "airplane_automobile": (0, 1)}.items():
    for severity in rotation_levels:
        example_to_rotation[f"{family}_{severity}"] = f"pair_{pair[0]}_{pair[1]}_{severity}"
comparison = rotation_examples.assign(
    rotation_scenario=rotation_examples.scenario.map(example_to_rotation)
).merge(
    rotation_split, left_on=rotation_keys + ["rotation_scenario"],
    right_on=rotation_keys + ["scenario"], suffixes=("_example", "_rotation"),
    validate="one_to_one",
)
assert len(comparison) == 126
rotation_example_error = max(
    float(np.abs(comparison[f"{metric}_example"] - comparison[f"{metric}_rotation"]).max())
    for metric in [*rotation_metrics, "coverage_gap"]
)
assert rotation_example_error < 1e-12

rotation_summary_metrics = [*rotation_metrics, "coverage_gap", "coverage_change_from_balanced"]
rotation_summary_keys = [
    "method", "model_key", "scenario", "pair_class_1", "pair_class_2",
    "pair_name", "severity", "focus_mass",
]
rotation_summary = rotation_split.groupby(rotation_summary_keys, as_index=False).agg(**{
    f"{metric}_{stat}": (metric, stat)
    for metric in rotation_summary_metrics for stat in ("mean", "std")
})
assert len(rotation_summary) == 816

rotation_overview_rows = []
for (method, model, severity), frame in rotation_summary[
    rotation_summary.severity != "balanced"
].groupby(["method", "model_key", "severity"], sort=True):
    balanced = rotation_summary[
        (rotation_summary.method == method) & (rotation_summary.model_key == model)
        & (rotation_summary.severity == "balanced")
    ].iloc[0]
    assert len(frame) == 45
    row = {
        "method": method, "model_key": model, "severity": severity,
        "focus_mass": rotation_levels[severity], "n_pairs": 45,
        "balanced_coverage_mean": float(balanced.empirical_coverage_mean),
        "pairs_below_nominal": int((frame.empirical_coverage_mean < 0.90).sum()),
        "pairs_below_balanced": int((frame.coverage_change_from_balanced_mean < -1e-12).sum()),
        "pairs_above_balanced": int((frame.coverage_change_from_balanced_mean > 1e-12).sum()),
    }
    for metric in rotation_metrics:
        values = frame[f"{metric}_mean"]
        row[f"{metric}_mean_over_pairs"] = float(values.mean())
        row[f"{metric}_min"] = float(values.min())
        row[f"{metric}_max"] = float(values.max())
        row[f"{metric}_p10"] = float(values.quantile(0.10))
        row[f"{metric}_p90"] = float(values.quantile(0.90))
        assert np.isclose(values.mean(), balanced[f"{metric}_mean"], rtol=0, atol=1e-12)
    rotation_overview_rows.append(row)
rotation_overview = pd.DataFrame(rotation_overview_rows)
assert len(rotation_overview) == 18
rotation_outputs = {
    "label_shift_rotation_target_priors.csv": rotation_priors,
    "label_shift_rotation_split_results.csv": rotation_split,
    "label_shift_rotation_summary_mean_std.csv": rotation_summary,
    "label_shift_rotation_overview.csv": rotation_overview,
}
for filename, frame in rotation_outputs.items():
    frame.to_csv(rotation_dir / filename, index=False)
rotation_manifest = {
    "analysis": "All unordered two-class majority-pair rotations",
    "class_pairs": [list(pair) for pair in rotation_pairs],
    "pair_count": 45, "scenario_count_including_balanced": 136,
    "focus_mass_by_severity": rotation_levels,
    "split_seeds": [42, 43, 44], "split_result_rows": 2448,
    "scenario_summary_rows": 816, "overview_rows": 18, "prior_rows": 1360,
    "input_sha256": {
        name: hashlib.sha256((rotation_dir / name).read_bytes()).hexdigest()
        for name in rotation_inputs
    },
    "output_sha256": {
        name: hashlib.sha256((rotation_dir / name).read_bytes()).hexdigest()
        for name in rotation_outputs
    },
    "original_example_max_absolute_error": rotation_example_error,
    "all_pair_mean_equals_balanced": True,
    "aggregation": "First average each scenario across the three overlapping splits; then summarize all 45 fixed pair means. Pair ranges and fractions are descriptive, not independent-sample confidence intervals.",
    "interpretation": "Equal enumeration gives every class the same average prior. Mean metrics over pairs equal balanced metrics at every severity; the range and frequency of lower coverage describe sensitivity to the emphasized classes.",
}
(rotation_dir / "label_shift_rotation_manifest.json").write_text(
    json.dumps(rotation_manifest, indent=2) + "\n", encoding="utf-8"
)
print("Full rotation: 45 pairs x 3 levels + balanced = 136 target priors.")
print("Saved 2,448 split results, 816 scenario summaries, 1,360 prior rows, and 18 overview rows.")
print(f"All 126 original example rows reproduce; maximum metric difference {rotation_example_error:.3g}.")
print("All-pair average-prior and balanced-metric identities: PASS.")
print(f"CPU reweighting and file export: {time.perf_counter() - rotation_started:.2f} seconds.")
display(rotation_overview.loc[
    rotation_overview.severity == "severe",
    ["method", "model_key", "empirical_coverage_mean_over_pairs",
     "empirical_coverage_min", "empirical_coverage_max", "pairs_below_nominal", "n_pairs"]
].round(4))


Full rotation: 45 pairs x 3 levels + balanced = 136 target priors.
Saved 2,448 split results, 816 scenario summaries, 1,360 prior rows, and 18 overview rows.
All 126 original example rows reproduce; maximum metric difference 4.44e-16.
All-pair average-prior and balanced-metric identities: PASS.
CPU reweighting and file export: 0.39 seconds.


,method,model_key,empirical_coverage_mean_over_pairs,empirical_coverage_min,empirical_coverage_max,pairs_below_nominal,n_pairs
2,APS,augmix_resnet18,0.9024,0.8971,0.9061,8,45
5,APS,standard_resnet18,0.9017,0.8862,0.9095,12,45
8,LAC,augmix_resnet18,0.9013,0.8479,0.9258,19,45
11,LAC,standard_resnet18,0.9016,0.8629,0.9207,16,45
14,RAPS,augmix_resnet18,0.9018,0.8959,0.9073,11,45
17,RAPS,standard_resnet18,0.9011,0.8849,0.9098,12,45


## Interpretation and completed outputs

Class proportions change aggregate coverage and efficiency through the class-conditional statistics. Each image's prediction set stays fixed. Coverage can rise or fall according to the emphasized classes, so read it with set size, empty-set rate, and accuracy. The usual exchangeability-based marginal guarantee does not automatically apply to a shifted evaluation population.

The complete design has 136 priors and 2,448 split-specific results. Its 816 scenario summaries retain the mean and sample SD across the three overlapping splits; the 18 overview rows describe all 45 pair means at each severity for every model/method. Four rotation CSVs and a hash manifest are saved under `results/`.

Equal averaging across all pairs returns the balanced prior by construction. The average therefore masks variation across majority-class identities; interpret it alongside pair ranges and the counts below nominal coverage. The two detailed example contrasts are a subset of these complete results.